# Phase 7 (Prior Case Retrieval) — Run #2: Phase-1 **Encoder-Decoder model 01**, its **bidirectional encoder** as a bi-encoder, with **IA³**

Same pipeline as run #1 (Dec-02 + LoRMA), so the two runs differ only in the **backbone** and the **adaptation technique**: identical data, views, token cache, BM25 lists, objective, fusion, model selection, analysis and exports. Edit **Cell 2** only.

### What run #1 taught us, and what changes here
| Run #1 finding | Consequence here |
|---|---|
| Zero-shot Phase-1 Dec-02 already reaches dense H@7 ≈ 229 (F1 0.25, 58 % of BM25); `last` pooling is worst and centering helps every pooling, most of all `last` | for a **bidirectional** encoder every position sees the whole chunk, so the causal argument for `wmean`/`last` disappears: the candidates are `mean` and `first` (the `[BOS]` state, which attends to every token, like BERT's `[CLS]`); centering is again measured |
| Training: dense 229 → 299 hits (+31 %), dense recall@300 0.857 ≈ BM25's 0.862; fused 395 → 405 (+10, gained 13 / lost 3) | the dense model finds mostly the **same** gold cases as BM25; the run is judged by the fused gain and by gained/lost |
| Training loss still falling at epoch 4 (1.97 → 1.06, in-batch acc 0.44 → 0.66) while dev dense hits plateaued from epoch 3 | the gap is generalisation, not optimisation: more and harder negatives per step help more than more steps |
| Peak GPU 3.1 GB with B = 16 | **B = 32**: twice the in-batch negatives at the same cost per epoch (compute scales with pairs, not with B). InfoNCE with `n` candidates bounds the mutual information by `log n`, so more candidates per query is a tighter, more informative objective |
| RRF (369) far below z-score fusion (405) | z-fusion keeps BM25's **confidence** (MRR 0.90: its top score is often far above the rest), RRF throws it away; kept as a check only |

### The backbone and the technique
* **Encoder of Enc-Dec 01**: 6 bidirectional layers, d = 512, RoPE, trained on 768-token inputs (our 511-token chunks are inside that). Padding needs a **key-padding mask** here (unlike the causal decoders, where right padding is invisible to real tokens).
* **IA³** (Liu et al. 2022): a learned vector rescales the keys, the values and the FFN hidden activation of every block, `k ← l_k ⊙ k`, `v ← l_v ⊙ v`, `h_ff ← l_ff ⊙ h_ff`, all initialised to 1 (identity at step 0). It is the lightest technique in the list (a few thousand parameters in total), so it tests how far *re-weighting existing features* goes for retrieval, versus LoRMA's rank-16 multiplicative updates in run #1. Rescaling keys changes which tokens a query attends to (the logits `q·(l_k⊙k) = Σ_i q_i l_{k,i} k_i`); rescaling values and FFN channels changes what is read out. It cannot create new directions in the residual stream, only amplify or damp existing ones; the identity-initialised `d × d` head (shared with run #1) is what can rotate the embedding space.
* IA³'s very few parameters need a higher learning rate (3e-3, as in Phase 4).

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle
from pathlib import Path
from dataclasses import dataclass, asdict, field, fields
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import copy, hashlib  # used by the Phase-7 cells

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu128


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p7_encdec_ia3"
METHOD     = "ia3"            # Phase-7 list: IA3
SMOKE_TEST = False            # True: SMOKE_STEPS steps, dev on SMOKE_DEV_Q queries (full dev pool), NO test / export
IA3_SITES  = ("k", "v", "ff") # keys, values, FFN hidden activation

# ---- Phase-1 checkpoint (caps_enc_dec_architecture_01.ipynb; encoder_causal=False) ------------------------------------------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-encdec-01-best/best.pt") if IS_KAGGLE else Path("./input/best_encdec.pt")

# ---- Data + audit artifacts (same as run #1) -----------------------------------------------------
HF_REPO        = "Exploration-Lab/CS-781-Capstone"
LOCAL_DATA_DIR = None
P7    = "Phase_7_PCR"
FILES = {"train": ("Train/train.json", "Train/train_candidates.json"),
         "dev":   ("Dev/dev.json",     "Dev/dev_candidates.json"),
         "test":  ("Test/test.json",   "Test/test_candidates.json")}
BM25_PATH   = Path("/kaggle/input/datasets/ascyrax/caps-p7-bm25/bm25_top300.json") if IS_KAGGLE else Path("./input/bm25_top300.json")
TOKCACHE_IN = Path("/kaggle/input/datasets/ascyrax/caps-p7-tokcache") if IS_KAGGLE else Path("./input/tokcache")   # written by run #1
N_PROC      = 4

# ---- Views (identical to run #1, so the token cache is reused) -----------------------------------
CHUNK_TOKENS   = 512          # [BOS] + 510 text tokens (+ [EOS] if APPEND_EOS)
MAX_DOC_CHUNKS = 16
QUERY_VIEW     = "ctx+head"
CTX_WINDOW     = 1
Q_MAX_CHUNKS   = 4
CIT_TAG        = "<CITATION>"
APPEND_EOS     = False        # the encoder was pre-trained on [BOS] + text spans

# ---- Encoder / scoring ----------------------------------------------------------------------------
POOLING    = "auto"           # "mean" | "first" ([BOS] state) | "auto" (best zero-shot pooling on dev)
SCORE_MODE = "auto"           # "mean" | "maxsim" | "auto"
USE_HEAD   = True             # d x d linear head, identity at init (same as run #1)
TAU        = 0.05
EMB_BATCH  = 96

# ---- Training -------------------------------------------------------------------------------------
EPOCHS        = 5             # IA3 converges more slowly than LoRMA; the 6-layer encoder is ~2x faster per epoch than Dec-02
BATCH_PAIRS   = 32            # run #1 peaked at 3.1 GB with 16 -> twice the in-batch negatives
C_TRAIN       = 4
QC_TRAIN      = 2
HN_TOP        = 50
FWD_MICRO     = 48
WARMUP_FRAC   = 0.05
GRAD_CLIP     = 1.0
GRAD_CKPT     = True
TIME_BUDGET_MIN = 360
HP = {"ia3": dict(lr=3e-3, head_lr=2e-4, wd=0.0)}

# ---- Fusion / evaluation --------------------------------------------------------------------------
K_EVAL     = 7
ALPHA_GRID = tuple(np.round(np.arange(0.0, 1.0001, 0.05), 2))
RRF_K      = 60
RECALL_KS  = (7, 20, 50, 100, 300)

# ---- Exports --------------------------------------------------------------------------------------
EXPORT_TEACHER = True
SAVE_SCORES    = True
SAVE_TOP       = 300

# ---- Smoke test -----------------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_DEV_Q = 40, 40

# ---- derived --------------------------------------------------------------------------------------
assert METHOD in HP and POOLING in ("mean", "first", "auto") and SCORE_MODE in ("mean", "maxsim", "auto")
assert QUERY_VIEW in ("ctx", "head", "ctx+head") and set(IA3_SITES) <= {"k", "v", "ff"} and IA3_SITES
BODY = CHUNK_TOKENS - 2                       # same chunk bodies as run #1 (token cache signature)
SEQ_LEN = CHUNK_TOKENS                        # used by the checkpoint-loading cell
MAX_DOC_TOK = MAX_DOC_CHUNKS * BODY
MAX_DOC_CHARS = int(MAX_DOC_TOK * 4.6)
RUN_DIR = WORK_DIR / "runs" / RUN_NAME; RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | {METHOD} sites={IA3_SITES} {HP[METHOD]} | chunks {CHUNK_TOKENS} x <= {MAX_DOC_CHUNKS} | query view {QUERY_VIEW} | "
      f"B={BATCH_PAIRS} pairs, C={C_TRAIN}, QC={QC_TRAIN}, tau={TAU} | epochs {EPOCHS} | smoke={SMOKE_TEST}")

RUN p7_encdec_ia3 | ia3 sites=('k', 'v', 'ff') {'lr': 0.003, 'head_lr': 0.0002, 'wd': 0.0} | chunks 512 x <= 16 | query view ctx+head | B=32 pairs, C=4, QC=2, tau=0.05 | epochs 5 | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

# Sentences are tokenised exactly as given (no clean_text / normalisation).
_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- the same guard the Phase-1
# Decoder-03 notebook uses. Pre-tokens longer than 1000 bytes are BPE-encoded in 1000-byte pieces. Normal text never has such
# chunks, so its tokenisation (and every character offset) is unchanged; byte-level BPE decodes the pieces to the exact same bytes.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_enc_dec_architecture_01.ipynb` (Cell 3)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# ===================== Cell 3: model =====================
@dataclass
class ModelConfig:
    vocab_size: int = 32000
    n_encoder_layers: int = 6
    n_decoder_layers: int = 6
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8
    head_dim: int = 64
    ff_mult: float = 8 / 3
    d_ff: Optional[int] = None
    max_seq_len: int = 1024          # RoPE table size; must cover encoder_len and decoder_len
    encoder_len: int = 768
    decoder_len: int = 256
    norm_type: str = "rmsnorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"           # rope | learned
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"         # swiglu | gelu
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True
    encoder_causal: bool = False     # False = bidirectional encoder
    mlm_head: bool = True            # BERT-style MLM head on the encoder (auxiliary objective)
    arch_name: str = "encdec_bidir_mlm_v2"

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim
        assert self.n_head % self.n_kv_head == 0
        assert self.max_seq_len >= max(self.encoder_len, self.decoder_len)
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g

class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.b = nn.Parameter(torch.zeros(d)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return ((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype) * self.g + self.b

def build_norm(cfg):
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(cfg.d_model, cfg.norm_eps)
    if cfg.norm_type == "layernorm":
        return LayerNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)


class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len, theta=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids=None):
        T = q.shape[2]
        if position_ids is None:
            cos = self.cos[:T][None, None]; sin = self.sin[:T][None, None]
        else:                                    # position_ids: [B, T] or [1, T]
            cos = self.cos[position_ids].unsqueeze(1); sin = self.sin[position_ids].unsqueeze(1)
        dt = q.dtype
        q = self._rotate(q.float(), cos, sin); k = self._rotate(k.float(), cos, sin)
        return q.to(dt), k.to(dt)


class Attention(nn.Module):
    # Self-attention (optionally causal, with KV cache) or cross-attention (context / precomputed static_kv).
    def __init__(self, cfg, causal):
        super().__init__()
        self.cfg, self.causal = cfg, causal
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, nh):
        B, T, _ = x.shape
        return x.view(B, T, nh, self.head_dim).transpose(1, 2)

    def project_kv(self, src):
        return self._split(self.k_proj(src), self.n_kv_head), self._split(self.v_proj(src), self.n_kv_head)

    def _attend(self, q, k, v, causal, key_mask):
        T, S = q.size(2), k.size(2)
        dp = self.cfg.attn_dropout if self.training else 0.0
        mask, sdpa_causal = None, False
        if causal and T > 1:
            if T == S and key_mask is None:
                sdpa_causal = True
            else:   # general (offset) causal mask: query i sees keys <= i + (S - T)
                mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril(diagonal=S - T).view(1, 1, T, S)
        if key_mask is not None:                 # key_mask: [B, S], True/1 = real token
            km = key_mask.to(torch.bool).view(key_mask.size(0), 1, 1, S)
            mask = km if mask is None else (mask & km)
        if self.cfg.use_sdpa:
            return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp, is_causal=sdpa_causal)
        scores = (q.float() @ k.float().transpose(-2, -1)) * self.scale
        if sdpa_causal:
            mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril().view(1, 1, T, S)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        p = F.dropout(F.softmax(scores, dim=-1), dp, self.training).to(q.dtype)
        return p @ v

    def forward(self, x, rope=None, position_ids=None, key_mask=None, kv_cache=None,
                context=None, static_kv=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        is_cross = (context is not None) or (static_kv is not None)
        if static_kv is not None:
            k, v = static_kv
        elif context is not None:
            k, v = self.project_kv(context)
        else:
            k, v = self.project_kv(x)
            if rope is not None:
                q, k = rope(q, k, position_ids=position_ids)
            if kv_cache is not None:
                k = torch.cat([kv_cache[0], k], dim=2)
                v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = None if is_cross else (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1); v = v.repeat_interleave(rep, dim=1)
        out = self._attend(q, k, v, causal=(self.causal and not is_cross), key_mask=key_mask)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class GELUMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))

def build_mlp(cfg):
    return SwiGLU(cfg) if cfg.mlp_type == "swiglu" else GELUMLP(cfg)


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=cfg.encoder_causal)
        self.norm2, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, rope=None, key_mask=None, position_ids=None):
        h, _ = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, key_mask=key_mask)
        x = x + h
        return x + self.mlp(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=True)
        self.norm2, self.cross_attn = build_norm(cfg), Attention(cfg, causal=False)
        self.norm3, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, enc_out=None, cross_kv=None, enc_mask=None, rope=None,
                position_ids=None, kv_cache=None):
        h, new_self = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, kv_cache=kv_cache)
        x = x + h
        if cross_kv is not None:
            h, _ = self.cross_attn(self.norm2(x), static_kv=cross_kv, key_mask=enc_mask)
        else:
            h, _ = self.cross_attn(self.norm2(x), context=enc_out, key_mask=enc_mask)
        x = x + h
        return x + self.mlp(self.norm3(x)), new_self


class Encoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_encoder_layers)])
        self.norm_f = build_norm(cfg)
    def forward(self, input_ids, attention_mask=None, position_ids=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        for blk in self.blocks:
            x = blk(x, rope=self.rope, key_mask=attention_mask, position_ids=position_ids)
        return self.norm_f(x)

class Decoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_decoder_layers)])
        self.norm_f = build_norm(cfg)

    def precompute_cross_kv(self, enc_out):
        # Cross-attention K/V depend only on the encoder output: compute once per encoder pass.
        return [blk.cross_attn.project_kv(enc_out) for blk in self.blocks]

    def forward(self, input_ids, enc_out=None, enc_mask=None, position_ids=None,
                kv_caches=None, cross_kvs=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, enc_out=enc_out,
                        cross_kv=cross_kvs[i] if cross_kvs is not None else None,
                        enc_mask=enc_mask, rope=self.rope, position_ids=position_ids,
                        kv_cache=kv_caches[i] if kv_caches is not None else None)
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class Seq2SeqLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.encoder = Encoder(cfg, self.tok_emb)
        self.decoder = Decoder(cfg, self.tok_emb)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        # BERT-style transform before the (tied) output projection for MLM.
        self.mlm_transform = (nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model, bias=True), nn.GELU(),
                                            build_norm(cfg)) if cfg.mlm_head else None)

    def encode(self, enc_ids, enc_mask=None):
        return self.encoder(enc_ids, attention_mask=enc_mask)

    def forward(self, enc_ids, dec_ids, enc_mask=None):
        enc_out = self.encode(enc_ids, enc_mask)
        h, _ = self.decoder(dec_ids, enc_out=enc_out, enc_mask=enc_mask)
        return self.lm_head(h)

    def mlm_logits_at(self, enc_hidden, select):
        # Project only the masked positions -> [N, V] (avoids a full [B, T, V] tensor).
        return self.lm_head(self.mlm_transform(enc_hidden[select]))


def init_weights(model, cfg):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    # GPT-2-style residual scaling, per stack: encoder blocks add 2 residual branches, decoder blocks add 3.
    s_enc = 1.0 / math.sqrt(2 * cfg.n_encoder_layers)
    s_dec = 1.0 / math.sqrt(3 * cfg.n_decoder_layers)
    with torch.no_grad():
        for blk in model.encoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_enc); blk.mlp.down_proj.weight.mul_(s_enc)
        for blk in model.decoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_dec); blk.cross_attn.o_proj.weight.mul_(s_dec)
            blk.mlp.down_proj.weight.mul_(s_dec)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())   # .parameters() de-duplicates tied weights
    emb = model.tok_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


# ---- Checkpoint helpers (with the causal/bidirectional guard) ----
def cfg_from_dict(d):
    d = dict(d)
    if "encoder_causal" not in d:
        raise ValueError("Checkpoint config has no 'encoder_causal' key. This is probably an old causal-encoder "
                         "checkpoint. Refusing to guess: the flag is a mask, not a weight, so a wrong guess would "
                         "load silently and run the encoder in a mode it was never trained in. Add "
                         "ckpt['config']['encoder_causal'] = True/False manually if you know which run it is.")
    d.setdefault("mlm_head", False)          # pre-v2 checkpoints have no MLM head
    d.setdefault("arch_name", "legacy")
    known = {f.name for f in fields(ModelConfig)}
    dropped = sorted(set(d) - known)
    if dropped:
        print("cfg_from_dict: ignoring unknown keys", dropped)
    return ModelConfig(**{k: v for k, v in d.items() if k in known})

def atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def load_model_from_ckpt(path, expect_encoder_causal=False):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = cfg_from_dict(ck["config"])
    if expect_encoder_causal is not None and cfg.encoder_causal != expect_encoder_causal:
        raise ValueError(f"{path}: encoder_causal={cfg.encoder_causal}, expected {expect_encoder_causal}")
    model = Seq2SeqLM(cfg)
    model.load_state_dict(ck["model"], strict=True)
    st = ck.get("state", {})
    print(f"Loaded {path} | arch={cfg.arch_name} | encoder_causal={cfg.encoder_causal} | "
          f"enc_len={cfg.encoder_len} dec_len={cfg.decoder_len} | step={st.get('step', ck.get('step'))} | "
          f"val_loss={st.get('val_loss', ck.get('val_loss'))}")
    return model.to(device).eval(), cfg, ck

_c = ModelConfig(vocab_size=VOCAB_SIZE)
_pc = count_params(Seq2SeqLM(_c))
print(f"Default config params: total={_pc['total']/1e6:.2f}M (embedding {_pc['embedding']/1e6:.2f}M)")
del _c

Default config params: total=61.49M (embedding 16.38M)


## Load the Phase-1 checkpoint

In [6]:
# Cell 12 — Load the Phase-1 encoder-decoder checkpoint and sanity-check its ENCODER
lm, lm_cfg, _ck = load_model_from_ckpt(CKPT_PATH, expect_encoder_causal=False)       # refuses a causal-encoder checkpoint
del _ck
pc = count_params(lm)
n_enc = sum(p.numel() for p in lm.encoder.parameters())                              # embeddings (shared) + encoder blocks + final norm
print(f"Phase-1 model | arch={lm_cfg.arch_name} | enc layers={lm_cfg.n_encoder_layers} dec layers={lm_cfg.n_decoder_layers} "
      f"d_model={lm_cfg.d_model} pos={lm_cfg.pos_type} | total {pc['total']/1e6:.2f}M | ENCODER-ONLY (used for CJPE) {n_enc/1e6:.2f}M")
assert SEQ_LEN <= lm_cfg.max_seq_len, "window longer than the RoPE table of this checkpoint"
if SEQ_LEN > lm_cfg.encoder_len:
    print(f"WARNING: sequence length {SEQ_LEN} > the encoder's trained length {lm_cfg.encoder_len} (RoPE extrapolation, expect some degradation)")

# Sanity: if the weights did not load properly, masked-token prediction would sit near ln(V) ~ 10.4.
MASK_ID = getattr(tokenizer, "special_to_id", {}).get("<mask>", 6)
if lm_cfg.mlm_head:
    _sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
    _x = torch.tensor([[BOS_ID] + tokenizer.encode(_sent)], device=device)
    _pos = [p for p in (4, 9, 13) if p < _x.size(1) - 1]
    _xm = _x.clone(); _xm[0, _pos] = MASK_ID
    _sel = torch.zeros_like(_xm, dtype=torch.bool); _sel[0, _pos] = True
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        _h = lm.encode(_xm, torch.ones_like(_xm))
        _lg = lm.mlm_logits_at(_h, _sel)
    _l = F.cross_entropy(_lg.float(), _x[0, _pos]).item()
    print(f"masked-LM loss on a legal sentence (3 masked tokens): {_l:.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
    assert _l < 9.0, "checkpoint looks untrained / mis-loaded"
else:
    print("this checkpoint has no MLM head; skipping the masked-LM sanity check")

Loaded /kaggle/input/datasets/ascyrax/caps-encdec-01-best/best.pt | arch=encdec_bidir_mlm_v2 | encoder_causal=False | enc_len=768 dec_len=256 | step=15488 | val_loss=1.9592824522405863
Phase-1 model | arch=encdec_bidir_mlm_v2 | enc layers=6 dec layers=6 d_model=512 pos=rope | total 61.49M | ENCODER-ONLY (used for CJPE) 35.66M
masked-LM loss on a legal sentence (3 masked tokens): 0.117  (ln V = 10.37)


## Phase-7 data (the audit's loader) and the BM25 lists
The loader is the audit's (Cell 4), unchanged. IDs are matched on a canonical key (leading zeros stripped); every output uses the pool's own ID string. The BM25 lists come from the audit's Cell 12 (`bm25_top300.json`, best configuration, no year filter).

In [7]:
# Cell 13 — Load the 6 files (audit loader) + BM25 lists
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

def fetch(rel):
    rel = f"{P7}/{rel}"
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet, or download {P7} and set LOCAL_DATA_DIR.") from ex

def read_json_any(path):
    txt = Path(path).read_text(encoding="utf-8")
    try: return json.loads(txt)
    except json.JSONDecodeError: return [json.loads(l) for l in txt.splitlines() if l.strip()]

ID_KEYS   = ("id", "query_id", "case_id", "candidate_id", "doc_id", "document_id", "qid", "cid")
TEXT_KEYS = ("text", "case_text", "document", "content", "sentences", "judgment", "judgement", "body")
GOLD_KEYS = ("relevant_candidates", "relevant", "relevant_ids", "positives", "labels", "gold", "citations", "cited")
def _first(e, keys, default=None):
    for k in keys:
        if k in e: return e[k]
    return default
def as_sents(v):
    if isinstance(v, str): return [s for s in v.split("\n")]
    out = []
    for x in v:
        if isinstance(x, list): out.extend(str(y) for y in x)
        elif isinstance(x, dict): out.append(str(_first(x, TEXT_KEYS + ("sentence",), "")))
        else: out.append(str(x))
    return out
def canon(x): s = str(x).strip(); return s.lstrip("0") or "0"
def unwrap(raw):
    if isinstance(raw, dict) and len(raw) == 1:
        v = next(iter(raw.values()))
        if isinstance(v, (list, dict)) and len(v) > 1: return v
    if isinstance(raw, dict) and isinstance(raw.get("data"), list): return raw["data"]
    return raw
def items_of(raw):
    raw = unwrap(raw)
    if isinstance(raw, dict):
        out = []
        for k, v in raw.items():
            if isinstance(v, dict): v = dict(v); v.setdefault("id", k)
            else: v = {"id": k, "text": v}
            out.append(v)
        return out
    return list(raw)

DOCS = {}                      # canonical id -> {"id": original string, "sents": [...]}, shared by all splits (pools overlap)
POOL, QRY = {}, {}             # split -> list of canonical ids (pool order) ; split -> list of query dicts
for split, (qf, cf) in FILES.items():
    raw_c = read_json_any(fetch(cf)); order, seen = [], set()
    for e in items_of(raw_c):
        k = canon(_first(e, ID_KEYS))
        if k in seen: continue
        seen.add(k); order.append(k)
        if k not in DOCS: DOCS[k] = dict(id=str(_first(e, ID_KEYS)), sents=as_sents(_first(e, TEXT_KEYS)))
    POOL[split] = order; del raw_c
    raw_q = read_json_any(fetch(qf)); qs = []
    pool_set = set(order)
    for i, e in enumerate(items_of(raw_q)):
        k = canon(_first(e, ID_KEYS, i))
        if k not in DOCS: DOCS[k] = dict(id=str(_first(e, ID_KEYS, i)), sents=as_sents(_first(e, TEXT_KEYS)))
        g = _first(e, GOLD_KEYS)
        gold = None
        if split != "test":
            g = [g] if isinstance(g, (str, int)) else (g or [])
            gold = []
            for x in g:
                c = canon(x)
                if c in pool_set and c not in gold: gold.append(c)
        qs.append(dict(key=k, id=str(_first(e, ID_KEYS, i)), gold=gold))
    QRY[split] = qs; del raw_q; gc.collect()
    print(f"{split:5s}: {len(qs)} queries | pool {len(order)} | queries inside their own pool: {sum(q['key'] in pool_set for q in qs)}"
          + (f" | gold pairs {sum(len(q['gold']) for q in qs)} | queries without gold {sum(not q['gold'] for q in qs)}" if split != "test" else ""))
print("distinct documents over all splits:", len(DOCS))

_bm = json.loads(BM25_PATH.read_text())
BM25_CFG, BM25 = _bm["config"], {s: {canon(q): [(canon(c), float(v)) for c, v in lst] for q, lst in d.items()} for s, d in _bm["lists"].items()}
del _bm
print("BM25 config:", BM25_CFG)
for s in FILES:
    miss = sum(q["key"] not in BM25[s] for q in QRY[s]); assert miss == 0, f"{s}: {miss} queries have no BM25 list"

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

train: 827 queries | pool 4320 | queries inside their own pool: 827 | gold pairs 5308 | queries without gold 10


Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

... (3 outputs left out)

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

test : 237 queries | pool 1727 | queries inside their own pool: 237
distinct documents over all splits: 5825
BM25 config: {'ng': 'bi', 'k1': 8.0, 'b': 1.0, 'mode': 'binary', 'H@7': 395, 'F1@7': 0.43913285158421345, 'recall@100': 0.7543679342240494, 'MRR': 0.8950390337803802, 'year_slack': None}


## Tokenisation, views, cache
Every distinct document is tokenised once (pools overlap) into its **document view** (head 2/3 + tail 1/3 of `MAX_DOC_CHARS` characters, then at most `MAX_DOC_CHUNKS × 510` tokens), and every query additionally into its **evidence view** (the `<CITATION>` sentences ± `CTX_WINDOW`, in document order, at most `Q_MAX_CHUNKS × 510` tokens). The cache is keyed by the view settings and written to `WORK_DIR/tokcache/`. **Upload that folder as the Kaggle dataset `caps-p7-tokcache`**: every later Phase-7 run uses the same tokenizer and reads it instead of re-tokenising (~5–15 min saved per run).

In [8]:
# Cell 14 — Tokenise (multiprocessing) + cache; chunking
def cut_text(t):
    if len(t) <= MAX_DOC_CHARS: return t
    h = 2 * MAX_DOC_CHARS // 3
    return t[:h] + "\n" + t[-(MAX_DOC_CHARS - h):]
def ctx_text(sents):
    hit = [j for j, s in enumerate(sents) if CIT_TAG in s]
    keep = sorted({k for j in hit for k in range(max(0, j - CTX_WINDOW), min(len(sents), j + CTX_WINDOW + 1))})
    return "\n".join(sents[k] for k in keep)

_W_ITEMS = None
def _tok_work(i):
    k, want_ctx = _W_ITEMS[i]; sents = DOCS[k]["sents"]
    doc = np.asarray(tokenizer.encode(cut_text("\n".join(sents)))[:MAX_DOC_TOK], dtype=np.uint16)
    ctx = None
    if want_ctx:
        t = ctx_text(sents)
        ctx = np.asarray(tokenizer.encode(t[: int(Q_MAX_CHUNKS * BODY * 4.6)])[: Q_MAX_CHUNKS * BODY], dtype=np.uint16) if t else np.zeros(0, np.uint16)
    return k, doc, ctx

SIG = hashlib.md5(json.dumps([CHUNK_TOKENS, MAX_DOC_CHUNKS, CTX_WINDOW, Q_MAX_CHUNKS, CIT_TAG, VOCAB_SIZE]).encode()).hexdigest()[:10]
CACHE_NAME = f"p7_tok_{SIG}.pkl"
cache_out = WORK_DIR / "tokcache" / CACHE_NAME; cache_out.parent.mkdir(parents=True, exist_ok=True)
TOKD, TOKQ = {}, {}
for p in (TOKCACHE_IN / CACHE_NAME, cache_out):
    if p.exists():
        TOKD, TOKQ = pickle.loads(p.read_bytes()); print(f"token cache loaded: {p} ({len(TOKD)} docs, {len(TOKQ)} query views)"); break
qkeys = {q["key"] for s in FILES for q in QRY[s]}
todo = [(k, k in qkeys) for k in DOCS if k not in TOKD or (k in qkeys and k not in TOKQ)]
if todo:
    t0 = time.time(); _W_ITEMS = todo
    with mp.get_context("fork").Pool(N_PROC) as pool:
        for k, d, c in tqdm(pool.imap_unordered(_tok_work, range(len(todo)), chunksize=4), total=len(todo), desc="tokenise"):
            TOKD[k] = d
            if c is not None: TOKQ[k] = c
    cache_out.write_bytes(pickle.dumps((TOKD, TOKQ), protocol=4))
    print(f"tokenised {len(todo)} documents in {time.time() - t0:.0f}s -> {cache_out} ({cache_out.stat().st_size / 1e6:.0f} MB)")

def chunks_of(ids, max_chunks):
    ids = ids[: max_chunks * BODY]
    ch = [ids[i:i + BODY] for i in range(0, len(ids), BODY)]
    if len(ch) > 1 and len(ch[-1]) < 32: ch = ch[:-1]          # a 5-token tail chunk is noise
    return ch or [np.zeros(0, np.uint16)]
def doc_chunks(k): return chunks_of(TOKD[k], MAX_DOC_CHUNKS)
def query_chunks(k):
    head = chunks_of(TOKD[k], 1)
    ctx = chunks_of(TOKQ[k], Q_MAX_CHUNKS) if len(TOKQ.get(k, ())) else []
    if QUERY_VIEW == "head": return chunks_of(TOKD[k], Q_MAX_CHUNKS)
    if QUERY_VIEW == "ctx": return ctx or head
    return (head + ctx)[:Q_MAX_CHUNKS]
_nq = [len(query_chunks(q["key"])) for q in QRY["train"]]; _nd = [len(doc_chunks(k)) for k in POOL["train"]]
print(f"train: query chunks/query mean {np.mean(_nq):.2f} | doc chunks/doc mean {np.mean(_nd):.2f} (cap {MAX_DOC_CHUNKS}) | "
      f"queries with an empty ctx view: {sum(len(TOKQ.get(q['key'], ())) == 0 for q in QRY['train'])}")

token cache loaded: /kaggle/input/datasets/ascyrax/caps-p7-tokcache/p7_tok_18962ef646.pkl (5825 docs, 1182 query views)
train: query chunks/query mean 3.70 | doc chunks/doc mean 9.98 (cap 16) | queries with an empty ctx view: 1


## The adapted model, embedding, scoring, evaluation, fusion

In [9]:
# Cell 15 — IA3 (the Phase-4 code, unchanged) + bi-encoder construction
class IA3Out(nn.Module):
    # y = l * (W x), l in R^{out_features}, initialised to 1 (keys / values).
    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x):
        y = self.base(x)
        return y * self.l.to(y.dtype)

class IA3In(nn.Module):
    # y = W (l * x), l in R^{in_features}, initialised to 1. On down_proj this scales the FFN hidden activation (l_ff).
    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x):
        return self.base(x * self.l.to(x.dtype))

def apply_adapter(module, targets, make):
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n

class BiEncoder(nn.Module):
    def __init__(self, enc, d):
        super().__init__()
        self.enc = enc
        self.head = nn.Linear(d, d, bias=False) if USE_HEAD else None
        if USE_HEAD:
            with torch.no_grad(): self.head.weight.copy_(torch.eye(d))

def build_model():
    # Fresh copy of the Phase-1 ENCODER; everything frozen except the IA3 vectors and the identity-initialised head.
    enc = copy.deepcopy(lm.encoder).cpu()
    for p in enc.parameters(): p.requires_grad = False
    ref = copy.deepcopy(enc)
    n_out = apply_adapter(enc, tuple(t for s_, t in (("k", "k_proj"), ("v", "v_proj")) if s_ in IA3_SITES), IA3Out)
    n_in = apply_adapter(enc, ("down_proj",), IA3In) if "ff" in IA3_SITES else 0
    n_ia3 = sum(p.numel() for p in enc.parameters() if p.requires_grad)
    print(f"IA3: {n_out} key/value scaling vectors + {n_in} FFN-hidden vectors = {n_ia3:,} parameters | sites={IA3_SITES}")
    _ids = torch.randint(5, VOCAB_SIZE, (2, 48)); _m = torch.ones_like(_ids); _m[1, 30:] = 0
    with torch.no_grad():
        a = enc.eval()(_ids, attention_mask=_m); b = ref.eval()(_ids, attention_mask=_m)
    err = (a - b).abs().max().item()
    print(f"identity-at-init check: max|adapted - Phase-1| = {err:.2e}")
    assert err < 1e-4, "adapter does not start from the Phase-1 function"
    del ref
    model = BiEncoder(enc, lm_cfg.d_model).to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"params: total {tot/1e6:.2f}M (encoder incl. embeddings + head) | trainable {tr/1e6:.4f}M ({100*tr/tot:.3f}%) = "
          f"IA3 {n_ia3:,} + head {tr - n_ia3:,}")
    model.n_total, model.n_trainable, model.n_peft = tot, tr, n_ia3
    return model

def backbone_hidden(model, ids, mask):
    # Phase-1 Encoder.forward (bidirectional, key-padding mask) with optional activation checkpointing (training only).
    enc = model.enc; N, T = ids.shape
    x = enc.tok_emb(ids); pos = None
    if enc.pos_emb is not None:
        pos = torch.arange(T, device=ids.device)[None]; x = x + enc.pos_emb(pos)
    x = enc.emb_drop(x)
    for blk in enc.blocks:
        if model.training and GRAD_CKPT and torch.is_grad_enabled():
            x = checkpoint(lambda t, blk=blk: blk(t, rope=enc.rope, key_mask=mask, position_ids=pos), x, use_reentrant=False)
        else:
            x = blk(x, rope=enc.rope, key_mask=mask, position_ids=pos)
    return enc.norm_f(x)

In [10]:
# Cell 16 — Pooling, chunk embedding, document scoring (mean / maxsim), evaluation, fusion
def make_batch(seqs):
    # seqs: list of uint16 arrays (chunk bodies). Returns ids [N, T] right-padded: [BOS] body ([EOS]), lengths [N].
    extra = 2 if APPEND_EOS else 1
    lens = [len(s) + extra for s in seqs]; T = max(lens)
    ids = torch.full((len(seqs), T), PAD_ID, dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, 0] = BOS_ID; ids[i, 1:1 + len(s)] = torch.from_numpy(s.astype(np.int64))
        if APPEND_EOS: ids[i, 1 + len(s)] = EOS_ID
    return ids.to(device, non_blocking=True), torch.tensor(lens, device=device)

def pool_hidden(h, lens, how):
    # h [N, T, d] (right padding), lens [N] -> [N, d]
    N, T, _ = h.shape
    t = torch.arange(T, device=h.device).unsqueeze(0)
    m = (t < lens.unsqueeze(1)).to(h.dtype)
    if how == "mean":  w = m
    elif how == "wmean": w = m * (t + 1).to(h.dtype)                 # SGPT position weights
    elif how == "last":  return h[torch.arange(N, device=h.device), lens - 1]
    elif how == "first": return h[:, 0]                                # [BOS]: attends to every token in a bidirectional encoder
    else: raise ValueError(how)
    return (h * w.unsqueeze(-1)).sum(1) / w.sum(1, keepdim=True)

def encode(model, ids, lens, poolings):
    mask = (torch.arange(ids.shape[1], device=ids.device).unsqueeze(0) < lens.unsqueeze(1)).long()     # key-padding mask
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        h = backbone_hidden(model, ids, mask)
    out = {}
    for p in poolings:
        e = pool_hidden(h.float(), lens, p)
        if getattr(model, "head", None) is not None: e = model.head(e)
        out[p] = F.normalize(e, dim=-1)
    return out

@torch.no_grad()
def embed_keys(model, keys, chunk_fn, poolings):
    # -> {pooling: [n_chunks, d] float32 (GPU)}, owner [n_chunks] (index into keys), sorted by owner
    model.eval()
    seqs, owner = [], []
    for i, k in enumerate(keys):
        for c in chunk_fn(k): seqs.append(c); owner.append(i)
    order = np.argsort([len(s) for s in seqs], kind="stable")
    E = {p: torch.empty(len(seqs), lm_cfg.d_model, device=device) for p in poolings}
    for b in range(0, len(order), EMB_BATCH):
        idx = order[b:b + EMB_BATCH]
        ids, lens = make_batch([seqs[j] for j in idx])
        out = encode(model, ids, lens, poolings)
        for p in poolings: E[p][torch.as_tensor(idx, device=device)] = out[p]
    return E, torch.tensor(owner, device=device)

def embed_split(model, split, poolings, q_subset=None):
    qs = QRY[split] if q_subset is None else QRY[split][:q_subset]
    t0 = time.time()
    Eq, oq = embed_keys(model, [q["key"] for q in qs], query_chunks, poolings)
    Ed, od = embed_keys(model, POOL[split], doc_chunks, poolings)
    return dict(Eq=Eq, oq=oq, Ed=Ed, od=od, nq=len(qs), nd=len(POOL[split]), secs=time.time() - t0, n_chunks=len(od) + len(oq))

def seg_mean(E, owner, n):
    s = torch.zeros(n, E.shape[1], device=E.device).index_add_(0, owner, E)
    return F.normalize(s, dim=-1)

@torch.no_grad()
def dense_scores(emb, pooling, mode, center=False, block=256):
    Eq, Ed, oq, od = emb["Eq"][pooling], emb["Ed"][pooling], emb["oq"], emb["od"]
    if center:
        mu = Ed.mean(0, keepdim=True); Eq = F.normalize(Eq - mu, dim=-1); Ed = F.normalize(Ed - mu, dim=-1)
    nq, nd = emb["nq"], emb["nd"]
    if mode == "mean":
        S = seg_mean(Eq, oq, nq) @ seg_mean(Ed, od, nd).T
    else:                                               # maxsim: mean over query chunks of the max over candidate chunks
        S = torch.zeros(nq, nd, device=device); cnt = torch.zeros(nq, device=device)
        for b in range(0, len(oq), block):
            sim = Eq[b:b + block] @ Ed.T                                        # [qb, n_doc_chunks]
            mx = torch.full((sim.shape[0], nd), -2.0, device=device).scatter_reduce(1, od.expand(sim.shape[0], -1), sim, "amax")
            S.index_add_(0, oq[b:b + block], mx); cnt.index_add_(0, oq[b:b + block], torch.ones(len(mx), device=device))
        S = S / cnt.clamp(min=1).unsqueeze(1)
    return S.float().cpu().numpy()

def self_mask(S, split, nq=None):
    col = {k: j for j, k in enumerate(POOL[split])}; S = S.copy()
    for i, q in enumerate(QRY[split][: S.shape[0]]):
        if q["key"] in col and not (q["gold"] and q["key"] in q["gold"]): S[i, col[q["key"]]] = -np.inf
    return S

def bm25_matrix(split, nq=None):
    col = {k: j for j, k in enumerate(POOL[split])}; qs = QRY[split][:nq] if nq else QRY[split]
    S = np.zeros((len(qs), len(POOL[split])), dtype=np.float32)
    for i, q in enumerate(qs):
        lst = BM25[split][q["key"]]; lo = min(v for _, v in lst) if lst else 0.0
        S[i, :] = lo - 1.0
        for c, v in lst:
            if c in col: S[i, col[c]] = v
    return S

def topk(S, k):
    k = min(k, S.shape[1]); idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    o = np.argsort(-np.take_along_axis(S, idx, 1), axis=1); return np.take_along_axis(idx, o, 1)
def gold_cols(split, nq=None):
    col = {k: j for j, k in enumerate(POOL[split])}; qs = QRY[split][:nq] if nq else QRY[split]
    return [set(col[g] for g in q["gold"]) for q in qs]
def evaluate(S, split, ks=(K_EVAL,), recall_ks=RECALL_KS):
    gc_ = gold_cols(split, S.shape[0]); N = len(gc_); G = sum(len(g) for g in gc_)
    T = topk(S, max(max(ks), max(recall_ks))); out = {}
    for k in ks:
        H = sum(len(set(T[i, :k].tolist()) & gc_[i]) for i in range(N))
        out[f"H@{k}"] = H; out[f"P@{k}"] = H / (k * N); out[f"R@{k}"] = H / max(G, 1); out[f"F1@{k}"] = 2 * H / (k * N + G)
    for k in recall_ks: out[f"recall@{k}"] = sum(len(set(T[i, :k].tolist()) & gc_[i]) for i in range(N)) / max(G, 1)
    rr = [next((1 / (j + 1) for j, c in enumerate(T[i].tolist()) if c in gc_[i]), 0.0) for i in range(N)]
    out["MRR"] = float(np.mean(rr)); return out

def znorm(S):
    S = S.astype(np.float64); f = np.isfinite(S)
    m = np.where(f, S, 0).sum(1, keepdims=True) / f.sum(1, keepdims=True)
    v = np.where(f, (S - m) ** 2, 0).sum(1, keepdims=True) / f.sum(1, keepdims=True)
    return np.where(f, (S - m) / np.sqrt(v + 1e-12), -np.inf)
def rrf(*Ss):
    out = np.zeros_like(Ss[0], dtype=np.float64)
    for S in Ss:
        r = np.empty_like(S, dtype=np.int64); o = np.argsort(-S, axis=1); r[np.arange(len(S))[:, None], o] = np.arange(S.shape[1])
        out += 1.0 / (RRF_K + 1 + r)
    return out
def fuse_search(Sd, Sb, split):
    # Sd, Sb already self-masked. Returns best (alpha, eval) by H@7 (ties: recall@100), and the whole curve.
    zd, zb = znorm(Sd), znorm(Sb); curve = []
    for a in ALPHA_GRID:
        e = evaluate(a * np.nan_to_num(zd, neginf=-1e9) + (1 - a) * np.nan_to_num(zb, neginf=-1e9), split)
        curve.append((float(a), e["H@7"], e["F1@7"], e["recall@100"]))
    best = max(curve, key=lambda t: (t[1], t[3], -abs(t[0] - 0.5)))
    return best, curve
def fused(Sd, Sb, alpha):
    zd, zb = znorm(Sd), znorm(Sb)
    return alpha * np.nan_to_num(zd, neginf=-1e9) + (1 - alpha) * np.nan_to_num(zb, neginf=-1e9)

## Zero-shot transfer of the Phase-1 encoder (before any training) — compare with run #1's table
One pass over dev produces both poolings (they share the hidden states). For each pooling and score mode we report **dense-only** dev metrics, the **best fusion with BM25**, and the effect of **centering** (subtracting the pool's mean chunk vector before the cosine). Centering targets *anisotropy*: language-model hidden states share a large common direction, so all cosines are close to each other and dominated by it (Ethayarajh 2019); removing the mean exposes the directions that actually differ between documents. If centering helps a lot zero-shot, that is direct evidence of anisotropy in the Phase-1 model.

In [11]:
# Cell 17 — Build the model; BM25-only dev reference; zero-shot table; choose POOLING
seed_everything(SEED)
model = build_model()
DEV_Q = SMOKE_DEV_Q if SMOKE_TEST else None
Sb_dev = self_mask(bm25_matrix("dev", DEV_Q), "dev")
e_b = evaluate(Sb_dev, "dev")
print(f"BM25 only (dev{'[:%d]' % DEV_Q if DEV_Q else ''}): H@7 {e_b['H@7']} F1@7 {e_b['F1@7']:.4f} recall@100 {e_b['recall@100']:.3f} MRR {e_b['MRR']:.3f}")

POOLS_ALL = ("mean", "first")
emb0 = embed_split(model, "dev", POOLS_ALL, DEV_Q)
print(f"dev embedded: {emb0['n_chunks']} chunks in {emb0['secs']:.0f}s")
ZS = []
for p in POOLS_ALL:
    for mode in ("mean", "maxsim"):
        for center in (False, True):
            Sd = self_mask(dense_scores(emb0, p, mode, center), "dev")
            e = evaluate(Sd, "dev"); (a, H, f1, r100), _ = fuse_search(Sd, Sb_dev, "dev")
            ZS.append(dict(pooling=p, mode=mode, center=center, H7=e["H@7"], F1=e["F1@7"], r100=e["recall@100"], MRR=e["MRR"],
                           fused_H7=H, fused_F1=f1, alpha=a))
            print(f"  zero-shot {p:5s} {mode:6s} center={center!s:5s}: dense H@7 {e['H@7']:4d} F1 {e['F1@7']:.4f} r@100 {e['recall@100']:.3f} | "
                  f"fused H@7 {H:4d} F1 {f1:.4f} (alpha {a})")
if POOLING == "auto":
    POOLING = max(POOLS_ALL, key=lambda p: max(r["H7"] for r in ZS if r["pooling"] == p))
print("POOLING ->", POOLING)
del emb0; gc.collect(); torch.cuda.empty_cache() if device.type == "cuda" else None

IA3: 12 key/value scaling vectors + 6 FFN-hidden vectors = 14,592 parameters | sites=('k', 'v', 'ff')
identity-at-init check: max|adapted - Phase-1| = 0.00e+00
params: total 35.93M (encoder incl. embeddings + head) | trainable 0.2767M (0.770%) = IA3 14,592 + head 262,144
BM25 only (dev): H@7 395 F1@7 0.4391 recall@100 0.754 MRR 0.895
dev embedded: 11164 chunks in 34s
  zero-shot mean  mean   center=False: dense H@7  185 F1 0.2057 r@100 0.544 | fused H@7  398 F1 0.4425 (alpha 0.15)
  zero-shot mean  mean   center=True : dense H@7  201 F1 0.2235 r@100 0.601 | fused H@7  398 F1 0.4425 (alpha 0.25)
  zero-shot mean  maxsim center=False: dense H@7  183 F1 0.2034 r@100 0.545 | fused H@7  398 F1 0.4425 (alpha 0.2)
  zero-shot mean  maxsim center=True : dense H@7  190 F1 0.2112 r@100 0.569 | fused H@7  397 F1 0.4414 (alpha 0.1)
  zero-shot first mean   center=False: dense H@7   49 F1 0.0545 r@100 0.276 | fused H@7  397 F1 0.4414 (alpha 0.25)
  zero-shot first mean   center=True : dense H@7   4

## Training
Each step: `BATCH_PAIRS` (query, positive) pairs from a shuffled list of all train gold pairs (an epoch visits every gold pair once, so queries with many citations get proportionally more steps — consistent with the micro-averaged metric). For each pair one hard negative is drawn from the query's BM25 top-`HN_TOP` minus its gold and itself. Every document contributes `C_TRAIN` random chunks and every query `QC_TRAIN` random evidence-view chunks (fresh random choices every step = augmentation). Embeddings are the normalised means of the chunk vectors. The forward runs in pieces of `FWD_MICRO` sequences with activation checkpointing; the loss needs all `2B` documents together, so the backward is one call.

In [12]:
# Cell 18 — Training loop (multi-positive InfoNCE with false-negative masking), per-epoch dev evaluation
TRQ = [q for q in QRY["train"] if q["gold"]]
TR_PAIRS = [(qi, g) for qi, q in enumerate(TRQ) for g in q["gold"]]
TR_GOLD = [set(q["gold"]) for q in TRQ]
TR_HN = [[c for c, _ in BM25["train"][q["key"]][:HN_TOP] if c not in TR_GOLD[qi] and c != q["key"]] for qi, q in enumerate(TRQ)]
print(f"train: {len(TRQ)} queries with gold | {len(TR_PAIRS)} (query, positive) pairs | queries without a hard negative: {sum(not h for h in TR_HN)}")

def sample_chunks(ch, n, rng): return ch if len(ch) <= n else [ch[i] for i in sorted(rng.sample(range(len(ch)), n))]

def step_loss(model, batch, rng):
    # batch: list of (qi, pos_key). Returns loss, stats.
    docs = []
    for qi, pk in batch:
        docs.append(pk)
        hn = TR_HN[qi]; docs.append(rng.choice(hn) if hn else rng.choice(POOL["train"]))
    seqs, owner = [], []
    for i, (qi, _) in enumerate(batch):
        for c in sample_chunks(query_chunks(TRQ[qi]["key"]), QC_TRAIN, rng): seqs.append(c); owner.append(i)
    nQ = len(batch)
    for j, k in enumerate(docs):
        for c in sample_chunks(doc_chunks(k), C_TRAIN, rng): seqs.append(c); owner.append(nQ + j)
    embs = []
    for b in range(0, len(seqs), FWD_MICRO):
        ids, lens = make_batch(seqs[b:b + FWD_MICRO]); embs.append(encode(model, ids, lens, (POOLING,))[POOLING])
    E = torch.cat(embs); own = torch.tensor(owner, device=device)
    V = seg_mean(E, own, nQ + len(docs)); Q, D = V[:nQ], V[nQ:]
    logits = (Q @ D.T) / TAU                                                  # [B, 2B]
    mask = torch.zeros_like(logits, dtype=torch.bool)
    for i, (qi, _) in enumerate(batch):
        for j, k in enumerate(docs):
            if j != 2 * i and (k in TR_GOLD[qi] or k == TRQ[qi]["key"]): mask[i, j] = True
    logits = logits.masked_fill(mask, float("-inf"))
    target = torch.arange(nQ, device=device) * 2
    loss = F.cross_entropy(logits.float(), target)
    with torch.no_grad():
        acc = (logits.argmax(1) == target).float().mean().item()
        gap = (logits[torch.arange(nQ), target] - logits[torch.arange(nQ), target + 1]).mean().item() * TAU   # cos(q,pos) - cos(q,hard neg)
    return loss, dict(acc=acc, gap=gap, masked=int(mask.sum().item()), seqs=len(seqs))

def param_groups(model):
    hp = HP[METHOD]
    head = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("head.")]
    rest = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head.")]
    g = [dict(params=rest, lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"])]
    if head: g.append(dict(params=head, lr=hp["head_lr"], base_lr=hp["head_lr"], weight_decay=0.0))
    return g

def dev_eval(model, emb=None):
    emb = emb or embed_split(model, "dev", (POOLING,), DEV_Q)
    out = {}
    for mode in ("mean", "maxsim"):
        Sd = self_mask(dense_scores(emb, POOLING, mode), "dev")
        e = evaluate(Sd, "dev"); (a, H, f1, r100), _ = fuse_search(Sd, Sb_dev, "dev")
        out[mode] = dict(H7=e["H@7"], F1=e["F1@7"], r100=e["recall@100"], fused_H7=H, fused_F1=f1, alpha=a)
    return out, emb

def train_model(model, epochs, max_steps=None):
    opt = torch.optim.AdamW(param_groups(model), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]
    spe = math.ceil(len(TR_PAIRS) / BATCH_PAIRS); total = epochs * spe
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm))))
    print(f"{spe} steps/epoch | total {total} steps | warmup {warm}")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, step, done, t_train = [], 0, False, time.time()
    best = dict(score=None, epoch=0, state=None)
    for ep in range(1, epochs + 1):
        rng = random.Random(SEED + ep); pairs = TR_PAIRS[:]; rng.shuffle(pairs)
        model.train(); t0 = time.time(); L, A, Gp = [], [], []
        for b in range(0, len(pairs), BATCH_PAIRS):
            loss, st = step_loss(model, pairs[b:b + BATCH_PAIRS], rng)
            m = lr_mult(step)
            for g in opt.param_groups: g["lr"] = g["base_lr"] * m
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
            L.append(loss.item()); A.append(st["acc"]); Gp.append(st["gap"])
            if step == 10:
                per = (time.time() - t0) / 10
                print(f"  >> speed {per:.2f} s/step ({st['seqs']} seqs) | projected {per * spe / 60:.1f} min/epoch"
                      + (f" | peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if device.type == "cuda" else ""))
            if step % 50 == 0:
                print(f"  ep {ep} step {step}/{total} | loss {np.mean(L[-50:]):.4f} | in-batch acc {np.mean(A[-50:]):.3f} | "
                      f"cos(pos)-cos(hn) {np.mean(Gp[-50:]):+.4f} | lr x{m:.3f} | {time.time() - t0:.0f}s")
            if max_steps and step >= max_steps: done = True; break
            if TIME_BUDGET_MIN and (time.time() - t_train) / 60 > TIME_BUDGET_MIN:
                print(f"  TIME_BUDGET_MIN={TIME_BUDGET_MIN} reached -> stopping"); done = True; break
        ev, _ = dev_eval(model)
        score = max(ev[m]["fused_H7"] for m in ev) + 1e-3 * max(ev[m]["H7"] for m in ev)
        rec = dict(epoch=ep, loss=float(np.mean(L)), acc=float(np.mean(A)), gap=float(np.mean(Gp)), steps=step, secs=time.time() - t0, dev=ev)
        history.append(rec)
        print(f"epoch {ep}: loss {rec['loss']:.4f} acc {rec['acc']:.3f} | dev dense H@7 mean {ev['mean']['H7']} / maxsim {ev['maxsim']['H7']} | "
              f"fused H@7 mean {ev['mean']['fused_H7']} (a={ev['mean']['alpha']}) / maxsim {ev['maxsim']['fused_H7']} (a={ev['maxsim']['alpha']})")
        if best["score"] is None or score > best["score"]:
            best = dict(score=score, epoch=ep, state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
            torch.save(best, RUN_DIR / "best_trainable.pt")
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0)
    return history, best, stats

history, best, stats = train_model(model, EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)
print(f"\nbest epoch {best['epoch']} | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB")

train: 817 queries with gold | 5308 (query, positive) pairs | queries without a hard negative: 0
166 steps/epoch | total 830 steps | warmup 41
  >> speed 3.87 s/step (303 seqs) | projected 10.7 min/epoch | peak GPU 3.0 GB
  ep 1 step 50/830 | loss 3.2529 | in-batch acc 0.233 | cos(pos)-cos(hn) -0.0095 | lr x1.000 | 194s
  ep 1 step 100/830 | loss 2.6922 | in-batch acc 0.302 | cos(pos)-cos(hn) -0.0032 | lr x0.987 | 387s
  ep 1 step 150/830 | loss 2.6533 | in-batch acc 0.323 | cos(pos)-cos(hn) -0.0043 | lr x0.957 | 581s
epoch 1: loss 2.8441 acc 0.293 | dev dense H@7 mean 285 / maxsim 271 | fused H@7 mean 397 (a=0.45) / maxsim 397 (a=0.4)
  ep 2 step 200/830 | loss 2.5693 | in-batch acc 0.324 | cos(pos)-cos(hn) -0.0060 | lr x0.909 | 132s
  ep 2 step 250/830 | loss 2.4673 | in-batch acc 0.350 | cos(pos)-cos(hn) -0.0001 | lr x0.846 | 325s
  ep 2 step 300/830 | loss 2.3389 | in-batch acc 0.393 | cos(pos)-cos(hn) +0.0078 | lr x0.771 | 518s
epoch 2: loss 2.4526 acc 0.356 | dev dense H@7 mean 2

## Final dev evaluation and analysis
With the best epoch restored: dense-only and fused scores for both score modes (the better one becomes `SCORE_MODE`), the full `α` curve, reciprocal-rank fusion, the F1@K curve, the rank of every gold case under the fused score (versus under BM25 alone), how many gold cases the fused ranking **adds** to BM25's top 7 and how many it **loses**, and the per-query hit distribution.

In [13]:
# Cell 19 — Final dev evaluation: score mode, alpha curve, RRF, F1@K, gold ranks, gains vs BM25
ev_final, emb_dev = dev_eval(model)
if SCORE_MODE == "auto":
    SCORE_MODE = max(("mean", "maxsim"), key=lambda m: (ev_final[m]["fused_H7"], ev_final[m]["H7"]))
Sd_dev = self_mask(dense_scores(emb_dev, POOLING, SCORE_MODE), "dev")
(ALPHA, H_f, F1_f, r100_f), curve = fuse_search(Sd_dev, Sb_dev, "dev")
Sf_dev = fused(Sd_dev, Sb_dev, ALPHA)
e_d, e_f = evaluate(Sd_dev, "dev"), evaluate(Sf_dev, "dev", ks=tuple(range(1, 21)))
e_rrf = evaluate(rrf(Sd_dev, Sb_dev), "dev")
print(f"SCORE_MODE={SCORE_MODE} | POOLING={POOLING} | ALPHA={ALPHA}")
print(f"  BM25 only : H@7 {e_b['H@7']:4d} F1@7 {e_b['F1@7']:.4f} recall@100 {e_b['recall@100']:.3f} MRR {e_b['MRR']:.3f}")
print(f"  dense only: H@7 {e_d['H@7']:4d} F1@7 {e_d['F1@7']:.4f} recall@100 {e_d['recall@100']:.3f} MRR {e_d['MRR']:.3f}")
print(f"  fused     : H@7 {e_f['H@7']:4d} F1@7 {e_f['F1@7']:.4f} recall@100 {e_f['recall@100']:.3f} MRR {e_f['MRR']:.3f}  (P@7 {e_f['P@7']:.3f} R@7 {e_f['R@7']:.3f})")
print(f"  RRF       : H@7 {e_rrf['H@7']:4d} F1@7 {e_rrf['F1@7']:.4f} recall@100 {e_rrf['recall@100']:.3f}")
print("  alpha curve (alpha, H@7):", [(a, H) for a, H, _, _ in curve])
print("  F1@K (fused):", {k: round(e_f[f'F1@{k}'], 4) for k in range(1, 21)})

gcols = gold_cols("dev", Sd_dev.shape[0])
def ranks(S):
    o = np.argsort(-S, axis=1); r = np.empty_like(o); r[np.arange(len(o))[:, None], o] = np.arange(o.shape[1])
    return np.asarray([r[i, c] + 1 for i, g in enumerate(gcols) for c in g])
rb, rf, rd = ranks(Sb_dev), ranks(Sf_dev), ranks(Sd_dev)
print(f"  gold rank median: BM25 {np.median(rb):.0f} | dense {np.median(rd):.0f} | fused {np.median(rf):.0f}  "
      f"(top-100 share: {np.mean(rb <= 100):.3f} / {np.mean(rd <= 100):.3f} / {np.mean(rf <= 100):.3f})")
Tb, Tf = topk(Sb_dev, 7), topk(Sf_dev, 7)
gain = sum(len((set(Tf[i].tolist()) - set(Tb[i].tolist())) & gcols[i]) for i in range(len(gcols)))
loss_ = sum(len((set(Tb[i].tolist()) - set(Tf[i].tolist())) & gcols[i]) for i in range(len(gcols)))
hits = np.asarray([len(set(Tf[i].tolist()) & gcols[i]) for i in range(len(gcols))])
print(f"  vs BM25 top-7: gold cases gained {gain} | lost {loss_} | net {gain - loss_:+d}")
print(f"  hits@7 per query (fused): {dict(sorted(Counter(hits.tolist()).items()))} | zero-hit queries {(hits == 0).mean():.1%}")
FINAL = dict(score_mode=SCORE_MODE, pooling=POOLING, alpha=ALPHA, bm25=e_b, dense=e_d,
             fused={k: v for k, v in e_f.items() if k.endswith("@7") or k.startswith("recall") or k == "MRR"},
             f1_curve={k: e_f[f"F1@{k}"] for k in range(1, 21)}, rrf=e_rrf, alpha_curve=curve,
             gold_rank_median=dict(bm25=float(np.median(rb)), dense=float(np.median(rd)), fused=float(np.median(rf))),
             gained=gain, lost=loss_, zero_hit=float((hits == 0).mean()), by_epoch_modes=ev_final)

SCORE_MODE=maxsim | POOLING=mean | ALPHA=0.45
  BM25 only : H@7  395 F1@7 0.4391 recall@100 0.754 MRR 0.895
  dense only: H@7  281 F1@7 0.3124 recall@100 0.714 MRR 0.712
  fused     : H@7  400 F1@7 0.4447 recall@100 0.771 MRR 0.901  (P@7 0.484 R@7 0.411)
  RRF       : H@7  355 F1@7 0.3947 recall@100 0.777
  alpha curve (alpha, H@7): [(0.0, 395), (0.05, 396), (0.1, 396), (0.15, 395), (0.2, 395), (0.25, 394), (0.3, 394), (0.35, 397), (0.4, 398), (0.45, 400), (0.5, 399), (0.55, 399), (0.6, 396), (0.65, 394), (0.7, 395), (0.75, 395), (0.8, 387), (0.85, 378), (0.9, 363), (0.95, 344), (1.0, 281)]
  F1@K (fused): {1: 0.1852, 2: 0.2911, 3: 0.3617, 4: 0.3972, 5: 0.4261, 6: 0.4402, 7: 0.4447, 8: 0.4413, 9: 0.4413, 10: 0.4347, 11: 0.4271, 12: 0.4194, 13: 0.4124, 14: 0.4015, 15: 0.3945, 16: 0.3838, 17: 0.374, 18: 0.3629, 19: 0.3533, 20: 0.3444}
  gold rank median: BM25 12 | dense 25 | fused 12  (top-100 share: 0.754 / 0.714 / 0.771)
  vs BM25 top-7: gold cases gained 13 | lost 8 | net +5
  hits@7 

In [14]:
# Cell 20 — Test: embed, fuse with BM25, prediction.json -> zip (full runs only)
def validate_prediction(pred, split="test"):
    pool_ids = {DOCS[k]["id"] for k in POOL[split]}; qids = [q["id"] for q in QRY[split]]
    assert set(pred) == set(qids), f"missing/extra queries: {len(set(qids) - set(pred))} missing, {len(set(pred) - set(qids))} extra"
    for qid, lst in pred.items():
        assert isinstance(lst, list) and len(lst) == K_EVAL and len(set(lst)) == K_EVAL, f"{qid}: need {K_EVAL} unique ids"
        assert all(isinstance(c, str) and c in pool_ids for c in lst), f"{qid}: id not in the {split} pool"
    return True

TEST = {}
if not SMOKE_TEST:
    t0 = time.time()
    emb_test = embed_split(model, "test", (POOLING,))
    Sd_test = self_mask(dense_scores(emb_test, POOLING, SCORE_MODE), "test")
    Sb_test = self_mask(bm25_matrix("test"), "test")
    Sf_test = fused(Sd_test, Sb_test, ALPHA)
    T7 = topk(Sf_test, K_EVAL)
    PRED = {q["id"]: [DOCS[POOL["test"][j]]["id"] for j in T7[i].tolist()] for i, q in enumerate(QRY["test"])}
    validate_prediction(PRED)
    sub_dir = RUN_DIR / "submission"; sub_dir.mkdir(exist_ok=True)
    (sub_dir / "prediction.json").write_text(json.dumps(PRED, indent=1))
    zpath = RUN_DIR / f"submission_{RUN_NAME}.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z: z.write(sub_dir / "prediction.json", arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z: assert z.namelist() == ["prediction.json"]
    Tb7 = topk(Sb_test, K_EVAL)
    overlap = np.mean([len(set(T7[i].tolist()) & set(Tb7[i].tolist())) / K_EVAL for i in range(len(T7))])
    TEST = dict(seconds=time.time() - t0, embed_seconds=emb_test["secs"], chunks=emb_test["n_chunks"], top7_overlap_with_bm25=float(overlap))
    print(f"test: {len(PRED)} queries -> {zpath} | embed {emb_test['secs']:.0f}s ({emb_test['n_chunks']} chunks) | "
          f"top-7 overlap with BM25 {overlap:.2f}")
    if SAVE_SCORES:
        def top_list(S, split):
            T = topk(S, SAVE_TOP)
            return {q["id"]: [[DOCS[POOL[split][j]]["id"], round(float(S[i, j]), 5)] for j in T[i].tolist() if np.isfinite(S[i, j])]
                    for i, q in enumerate(QRY[split][: S.shape[0]])}
        (RUN_DIR / "dense_scores.json").write_text(json.dumps(dict(run=RUN_NAME, pooling=POOLING, mode=SCORE_MODE, alpha=ALPHA,
                                                                   dev=top_list(Sd_dev, "dev"), test=top_list(Sd_test, "test"))))
        print("saved dense top-%d scores for dev/test" % SAVE_TOP)

test: 237 queries -> /kaggle/working/runs/p7_encdec_ia3/submission_p7_encdec_ia3.zip | embed 65s (18838 chunks) | top-7 overlap with BM25 0.94
saved dense top-300 scores for dev/test


## KD teacher export (for the distillation run)
For every train query: the teacher's **raw cosine scores** (chosen pooling and score mode) over its BM25 top-300 ∪ its gold set, keyed by the pool's ID strings, together with `τ`. The student run turns them into a listwise target `softmax(s/τ)` over the same candidate list (and pairwise margins), so it learns the teacher's *ranking*, not just the binary gold labels. Only cosine similarities are exported, so the student can be any architecture with any embedding size.

In [15]:
# Cell 21 — Save trainable weights + log; teacher export
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer, np.bool_)): return o.item()
    if isinstance(o, float) and not math.isfinite(o): return None
    if isinstance(o, Path): return str(o)
    return o
torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, RUN_DIR / "trainable_final.pt")
log = dict(run=RUN_NAME, method=METHOD, hp=HP[METHOD], ia3_sites=IA3_SITES, params_peft=model.n_peft, view=dict(chunk=CHUNK_TOKENS, max_doc_chunks=MAX_DOC_CHUNKS, query_view=QUERY_VIEW,
           ctx_window=CTX_WINDOW, q_max_chunks=Q_MAX_CHUNKS), train=dict(epochs=EPOCHS, B=BATCH_PAIRS, C=C_TRAIN, QC=QC_TRAIN, hn_top=HN_TOP, tau=TAU),
           params_total=model.n_total, params_trainable=model.n_trainable, zero_shot=ZS, history=history, best_epoch=best["epoch"],
           stats=stats, final=FINAL, test=TEST, bm25_cfg=BM25_CFG)
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=1))

TEACHER = {}
if EXPORT_TEACHER and not SMOKE_TEST:
    t0 = time.time()
    emb_tr = embed_split(model, "train", (POOLING,))
    S_tr = self_mask(dense_scores(emb_tr, POOLING, SCORE_MODE), "train")
    col = {k: j for j, k in enumerate(POOL["train"])}
    exp = {}
    for i, q in enumerate(QRY["train"]):
        cands = [c for c, _ in BM25["train"][q["key"]]] + [g for g in (q["gold"] or [])]
        seen, row = set(), []
        for c in cands:
            if c in seen or c == q["key"] or c not in col: continue
            seen.add(c); row.append([DOCS[c]["id"], round(float(S_tr[i, col[c]]), 5)])
        exp[q["id"]] = row
    tpath = RUN_DIR / f"teacher_{RUN_NAME}.json"
    tpath.write_text(json.dumps(dict(run=RUN_NAME, tau=TAU, pooling=POOLING, mode=SCORE_MODE, alpha=ALPHA, scores=exp)))
    TEACHER = dict(seconds=time.time() - t0, queries=len(exp), mean_list=float(np.mean([len(v) for v in exp.values()])),
                   mb=tpath.stat().st_size / 1e6)
    print(f"teacher export: {TEACHER} -> {tpath}")
for f in sorted(RUN_DIR.iterdir()): print(f"   {f.name:40s} {f.stat().st_size / 1e6 if f.is_file() else 0:8.2f} MB")

teacher export: {'seconds': 160.07675981521606, 'queries': 827, 'mean_list': 300.9782345828295, 'mb': 6.208387} -> /kaggle/working/runs/p7_encdec_ia3/teacher_p7_encdec_ia3.json
   best_trainable.pt                            1.11 MB
   dense_scores.json                            2.66 MB
   run_log.json                                 0.01 MB
   submission                                   0.00 MB
   submission_p7_encdec_ia3.zip                 0.01 MB
   teacher_p7_encdec_ia3.json                   6.21 MB
   trainable_final.pt                           1.11 MB


In [16]:
# Cell 22 — PASTE TO CLAUDE
def _r(o, n=4):
    if isinstance(o, dict): return {k: _r(v, n) for k, v in o.items()}
    if isinstance(o, list): return [_r(v, n) for v in o]
    if isinstance(o, float): return round(o, n)
    return o
short = dict(run=RUN_NAME, smoke=SMOKE_TEST, hp=HP[METHOD], params_peft=model.n_peft, params_trainable=model.n_trainable, params_total=model.n_total,
             zero_shot=ZS, history=[{k: v for k, v in h.items()} for h in history], best_epoch=best["epoch"], stats=stats,
             final={k: v for k, v in FINAL.items() if k not in ("alpha_curve",)}, alpha_curve=[(a, H) for a, H, _, _ in FINAL["alpha_curve"]],
             test=TEST, teacher=TEACHER)
print("================ PASTE TO CLAUDE ================")
print(json.dumps(_r(_jsonable(short)), separators=(",", ":")))
print("=================================================")

================ PASTE TO CLAUDE ================
{"run":"p7_encdec_ia3","smoke":false,"hp":{"lr":0.003,"head_lr":0.0002,"wd":0.0},"params_peft":14592,"params_trainable":276736,"params_total":35934976,"zero_shot":[{"pooling":"mean","mode":"mean","center":false,"H7":185,"F1":0.2057,"r100":0.5437,"MRR":0.4748,"fused_H7":398,"fused_F1":0.4425,"alpha":0.15},{"pooling":"mean","mode":"mean","center":true,"H7":201,"F1":0.2235,"r100":0.6012,"MRR":0.4942,"fused_H7":398,"fused_F1":0.4425,"alpha":0.25},{"pooling":"mean","mode":"maxsim","center":false,"H7":183,"F1":0.2034,"r100":0.5447,"MRR":0.5524,"fused_H7":398,"fused_F1":0.4425,"alpha":0.2},{"pooling":"mean","mode":"maxsim","center":true,"H7":190,"F1":0.2112,"r100":0.5694,"MRR":0.568,"fused_H7":397,"fused_F1":0.4414,"alpha":0.1},{"pooling":"first","mode":"mean","center":false,"H7":49,"F1":0.0545,"r100":0.2765,"MRR":0.1772,"fused_H7":397,"fused_F1":0.4414,"alpha":0.25},{"pooling":"first","mode":"mean","center":true,"H7":44,"F1":0.0489,"r100":0.2